# Test Renderer

In [ ]:
import sys
import os
from pathlib import Path

# Add project root to path (adjust if notebook is in a subfolder)
project_root = Path.cwd().parent  # if notebook is in experiments/ or similar
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


import warnings

# Suppress the specific future warning from torchrl
warnings.filterwarnings(
    "ignore", 
    category=FutureWarning, 
    module="torchrl.modules.mcts.scores"
)

import torch
from torchrl.envs import EnvBase
from torchrl.data import (
    Composite, 
    Unbounded, 
    Bounded,
    Stacked
    
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

import numpy as np

# from urbanmarl.envs.urbanmarl_env import UrbanEnv
from urbanmarl.envs.base_env import UrbanEnv
from urbanmarl.envs.rendering import Urban3DRenderer, UrbanRenderConfig

In [ ]:
device

In [ ]:
# import matplotlib
# matplotlib.use('Agg')  # Headless backend
from matplotlib import pyplot as plt
# from mpl_toolkits.mplot3d import Axes3D
# from mpl_toolkits.mplot3d.art3d import Poly3DCollection

In [ ]:
config = {
    "num_uavs": 3,
    "num_ues": 20,
    "area_size": (500, 500),
    "max_time_slots": 50,
    "max_horizontal_speed": 49.0,
    "max_vertical_speed": 12.0,
    "max_transmit_power": 5.0,
    "frequency_ghz": 29.0,
    "g2a_bandwidth": 10e6,
    "noise_figure_db": 7.0,
    "agents": ["agent_0", "agent_1", "agent_2"]
    # "agents": ["uav_0", "uav_1", 'ue_0']
}
num_envs = 72
scenario = "uav_navigation"
# scenario = "uav_ue_los"
env = UrbanEnv(
    num_envs = num_envs, # batch_size=torch.Size([2])
    continuous_actions = True,
    seed=0,
    device=device,
    scenario=scenario,
    **config)

In [ ]:
def render(self, env: UrbanEnv, mode = 'rgb_array'):
    """Render UrbanMARL
    args:
        env: UrbanMARL environment
        mode: rgb_array or human

    return:
        
    """
    if not hasattr(self, 'renderer'):
        self.renderer = Urban3DRenderer()
    else:
        self.renderer._init_plot(env.volume_size)
    #
    if not hasattr(self, 'render_idx'):
        self.render_idx = np.random.randint(env.batch_size[0])
    #
    uav_positions = env.uav_agents_pos[self.render_idx].cpu()
    ue_positions = env.ue_user_pos[self.render_idx].cpu()
    los = env.uav_ue_los[self.render_idx]
    los_links = []
    
    if hasattr(env, "uav_ue_los"):
        for uav in range(env.n_uavs):
            for ue in range(env.n_ues):
                los_links.append({'source': uav_positions[uav], 'target': ue_positions[ue], 'los': los[uav, ue]})
    alpha, beta, gamma, e = env._env.info[self.render_idx][:4]
    title = f"UrbanMARL 3D Environment ({alpha:.2f}, {beta}, {gamma:.2f}, {e:.4f})"
    state = {
        'volume_size': env.volume_size,
        'buildings': env._env.building_data[self.render_idx],
        'uav_positions': uav_positions,
        'ue_positions': ue_positions,
        # 'base_station_positions': np.array([[250, 250, 30]]),
        'links': los_links,
        'title': title
    }
    
    return self.renderer.render(state, mode=mode)

n_rollout_steps = 3

tensordict = env.reset()
rollout = env.rollout(n_rollout_steps)

rgb_frame = render(env.scenario, env)

In [ ]:
img = env.scenario.render(env, mode='human')

In [ ]:
img = env.scenario.render(env, mode='rgb_array')
print(type(img), img.shape)
img_tensor = torch.from_numpy(img).permute(2, 0, 1)  # (H,W,3) -> (3,H,W)
print(type(img_tensor), img_tensor.shape)


In [ ]:
max_length_rollout_0 = 2
video_frames =[img_tensor for _ in range(100)]

video_frames = np.stack(video_frames[: max_length_rollout_0 - 1], axis=0)
vid = torch.tensor(
    np.transpose(video_frames, (0, 3, 1, 2)),
    dtype=torch.uint8,
).unsqueeze(0)
print(vid.dim())
print(vid.size())
print(vid.size(dim=2))

In [ ]:
n_rollout_steps = 3

tensordict = env.reset()
rollout = env.rollout(n_rollout_steps)

from tensordict import TensorDictBase
from torch import Tensor
# @staticmethod
def render_callback(experiment, env: UrbanEnv, data: TensorDictBase) -> Tensor:
    """
    BenchMARL callback for rendering during evaluation.
    Called at every step during evaluation to provide pixels for video logging.
    """
    # Call render on the underlying environment
    if hasattr(env, 'scenario'):
        # If wrapped, unwrap
        base = env.base_env
    else:
        base = env
        
    # Try to render
    try:
        pixels = base.render(mode="rgb_array")
        if pixels is None:
            # Fallback: return zeros
            return torch.zeros((480, 640, 3), dtype=torch.uint8)
        return torch.from_numpy(pixels)
    except Exception as e:
        # If rendering fails, return blank frame
        print(f"Render warning: {e}")
        return torch.zeros((480, 640, 3), dtype=torch.uint8)

In [ ]:
np.random.randint(10)

In [ ]:
rgb_frame = render(env.scenario, env)

In [ ]:
fig_size = (1200, 800)

renderer = Urban3DRenderer(UrbanRenderConfig(
    figsize=fig_size,
    show_trajectory=True,
    camera_elev=60,
    camera_azim=-120
))
# renderer = Urban3DRenderer()

urban_idx = np.random.randint(env.batch_size[0]) #np.random.permutation(env.batch_size[0])[0]

uav_positions = env.uav_agents_pos[urban_idx].cpu()
ue_positions = env.ue_user_pos[urban_idx].cpu()
los = env.uav_ue_los[urban_idx]
los_links = []

if hasattr(env, "uav_ue_los"):
    for uav in range(env.n_uavs):
        for ue in range(env.n_ues):
            los_links.append({'source': uav_positions[uav], 'target': ue_positions[ue], 'los': los[uav, ue]})
alpha, beta, gamma, e = env._env.info[urban_idx][:4]
title = f"UrbanMARL 3D Environment ({alpha:.2f}, {beta}, {gamma:.2f}, {e:.4f})"
state = {
    'volume_size': env.volume_size,
    'buildings': env._env.building_data[urban_idx],
    'uav_positions': uav_positions,
    'ue_positions': ue_positions,
    # 'base_station_positions': np.array([[250, 250, 30]]),
    'links': los_links,
    'title': title
}

rgb_frame = renderer.render(state, mode='rgb_array')


In [ ]:
# renderer = Urban3DRenderer(UrbanRenderConfig(
#     figsize=fig_size,
#     show_trajectory=True,
#     camera_elev=45,
#     camera_azim=120
# ))
# rgb_frame = renderer.render(state, mode='human')

In [ ]:
def render_2d(self, mode: str = "human"):
    import matplotlib.pyplot as plt


    fig, ax = plt.subplots(figsize=(7, 7))
    heat_map = self._env.height_maps[0].cpu().numpy()
    x_min = -float(self._env.volume_size[0]) / 2
    x_max = float(self._env.volume_size[0]) / 2
    y_min = -float(self._env.volume_size[1]) / 2
    y_max = float(self._env.volume_size[1]) / 2

    ax.imshow(
        heat_map,
        cmap="Greys",
        origin="lower",
        extent=[x_min, x_max, y_min, y_max],
        alpha=0.5,
    )
    ax.scatter(
        self.ue_user_pos[0, :, 0].cpu(),
        self.ue_user_pos[0, :, 1].cpu(),
        c="blue",
        s=20,
        label="UEs",
    )
    ax.scatter(
        self.uav_agents_pos[0, :, 0].cpu(),
        self.uav_agents_pos[0, :, 1].cpu(),
        c="red",
        marker="^",
        s=50,
        label="UAVs",
    )
    ax.set_title("UrbanMEC Environment")
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.legend(loc="upper right")
    ax.set_aspect("equal")

    if mode == "human":
        plt.show()
    return fig

video_frames = render_2d(env)

In [ ]:
np.transpose(video_frames, (0, 3, 1, 2))